# Run the experiments on Molab's GPU

Open the native `run_prerequisites_marimo.py` notebook on Molab, or use these equivalent JSON/Jupyter cells.

1. In Cell 2 leave SOURCE_ZIP=None: the notebook fetches published GitHub commit 864974efee26a5d09edf24d67f9e8aa3ed4f30b3, including the real recorded-data schema and bounded Zstandard decoder fixes. **No source ZIP or checksum upload is needed.** Optional offline ZIP mode still requires its matching receipt.
2. Set DATA to your existing remote pilot data, and choose a new RUN_ID. Keep PROJECT as a new remote folder. If the hosted session lost its data, supply BACKUP_ZIP plus its adjacent receipt to restore the saved pilot. Never reset an older experimental checkout to the new pin.
3. Set TRUST_PINNED_DINO=True only if you consent to loading the pinned released pickle. Set EXECUTE=True. Disable reactive auto-run while editing, then Run all.

The notebook prepares the CUDA environment **inside Molab**, then executes scripts sequentially with live output. Training fits small physical readouts; pretrained LeWM and DINO weights remain frozen. Recorded checks require RUN_RECORDED=True; the large optional dataset download additionally requires DOWNLOAD_RECORDED=True. Recorded raw controls outside the unit box are retained without clipping, with unchanged released model normalization; terminal rows never become outgoing transitions.

No editor refit or scientific confirmation is launched. Existing data and results are never overwritten. The completed six-experiment Molab run and its limitations are documented in MOLAB_RESULTS_20261101.md.

In [ ]:
from pathlib import Path
import importlib.util

SOURCE_ZIP = None  # Fetch the published pinned GitHub commit; no source upload needed.
PROJECT = Path('/marimo/world_models_diagnostics')  # NEW hosted source directory.
DATA = '/marimo/world_models/runs/pilot-gpu-01/data'  # Existing REMOTE data, or nested restored data.
BACKUP_ZIP = None  # Fresh session: absolute path to uploaded pilot backup + adjacent .zip.sha256 receipt.
EXISTING_PYTHON = None  # Bootstrap CUDA on Molab; or '/marimo/world_models/.venv/bin/python'.
CONFIG = 'configs/pilot.json'
RUN_ID = 'diagnostics-notebook-01'  # New for every attempt.
DEVICE = 'cuda'
EXECUTE = False
TRUST_PINNED_DINO = False
INSTALL_UPDATED_PACKAGE = True
RUN_RECORDED = False
DOWNLOAD_RECORDED = False
H5 = 'cache/datasets/lewm-pusht/pusht_expert_train.h5'
EPISODES = 128

In [ ]:
# Deploy only to a NEW hosted directory, or verify an identical previous deployment.
if not EXECUTE:
    raise RuntimeError('Review Cell 2 and set EXECUTE=True before remote setup')
import hashlib
import stat
import zipfile
from pathlib import PurePosixPath

if SOURCE_ZIP is None:
    import subprocess
    _repo = 'https://github.com/Wodlfvllf/world_models.git'
    _commit = '864974efee26a5d09edf24d67f9e8aa3ed4f30b3'
    if PROJECT.is_symlink():
        raise FileExistsError('PROJECT must not be a symlink')
    if not PROJECT.exists():
        PROJECT.mkdir(parents=True, exist_ok=False)
        print(f'Fetching published source {_commit} into {PROJECT}', flush=True)
        subprocess.run(['git', 'init', str(PROJECT)], check=True, timeout=60)
        subprocess.run(['git', '-C', str(PROJECT), 'remote', 'add', 'origin', _repo], check=True, timeout=60)
        subprocess.run(['git', '-C', str(PROJECT), 'fetch', '--progress', '--depth', '1', 'origin', _commit], check=True, timeout=300)
        subprocess.run(['git', '-C', str(PROJECT), 'checkout', '--detach', _commit], check=True, timeout=60)
    if not (PROJECT / '.git').is_dir():
        raise FileExistsError('Existing PROJECT is not the pinned checkout; choose a NEW PROJECT, do not delete results')
    _head = subprocess.check_output(['git', '-C', str(PROJECT), 'rev-parse', 'HEAD'], text=True, timeout=60).strip()
    _dirty = subprocess.check_output(['git', '-C', str(PROJECT), 'status', '--porcelain', '--untracked-files=no'], text=True, timeout=60).strip()
    if _head != _commit or _dirty:
        raise ValueError('Hosted source revision differs or tracked files changed; use a NEW PROJECT directory')
    if not (PROJECT / 'scripts/molab_setup.py').is_file():
        raise FileNotFoundError('Pinned checkout is missing the Molab setup script')
    print(f'Verified GitHub source: {_head}', flush=True)
else:
    _receipt = SOURCE_ZIP.with_suffix('.zip.sha256')
    if not SOURCE_ZIP.is_file() or not _receipt.is_file():
        raise FileNotFoundError('Optional source ZIP/receipt missing. Set SOURCE_ZIP=None to fetch pinned GitHub source, or upload both the ZIP and matching .zip.sha256 receipt at the configured path.')
    _expected = _receipt.read_text().split()[0]
    with SOURCE_ZIP.open('rb') as _stream:
        _actual = hashlib.file_digest(_stream, 'sha256').hexdigest()
    if _actual != _expected:
        raise ValueError('Source ZIP checksum mismatch; re-upload bundle and receipt')
    with zipfile.ZipFile(SOURCE_ZIP) as _archive:
        _names = _archive.namelist()
        if len(set(_names)) != len(_names) or 'scripts/molab_setup.py' not in _names:
            raise ValueError('Invalid source bundle')
        if sum(_info.file_size for _info in _archive.infolist()) > 50_000_000:
            raise ValueError('Expected a small source-only bundle, not a data/model backup')
        for _info in _archive.infolist():
            _parts = PurePosixPath(_info.filename)
            if (_parts.is_absolute() or '..' in _parts.parts or '\\' in _info.filename
                    or ':' in _info.filename or _info.is_dir() or stat.S_ISLNK(_info.external_attr >> 16)
                    or _parts.parts[0] in {'runs', 'cache', '.venv', '.git'}):
                raise ValueError('Unsafe source archive member')
        _marker = PROJECT / 'MOLAB-SOURCE.sha256'
        if PROJECT.exists() or PROJECT.is_symlink():
            if PROJECT.is_symlink() or not _marker.is_file() or _marker.read_text().strip() != _actual:
                raise FileExistsError('PROJECT must be new, or a previous extraction of this exact bundle')
            for _name in _names:
                _target = PROJECT / _name
                if (_target.is_symlink() or not _target.resolve().is_relative_to(PROJECT.resolve())
                        or _target.read_bytes() != _archive.read(_name)):
                    raise ValueError('Hosted source changed; use a new PROJECT directory')
        else:
            PROJECT.mkdir(parents=True, exist_ok=False)
            for _name in _names:
                _target = PROJECT / _name
                _target.parent.mkdir(parents=True, exist_ok=True)
                _target.write_bytes(_archive.read(_name))
            _marker.write_text(_actual + '\n')
source_ready = str(PROJECT)
print(f'Verified source deployed on Molab: {source_ready}', flush=True)

In [ ]:
# Bootstrap the environment ON MOLAB, optionally restore the receipt-verified pilot backup.
_spec = importlib.util.spec_from_file_location('molab_setup', PROJECT / 'scripts/molab_setup.py')
_module = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_module)
hosted_python, hosted_data = _module.prepare(source_ready, DATA, existing_python=EXISTING_PYTHON,
                                            backup=BACKUP_ZIP, device=DEVICE)

In [ ]:
if not EXECUTE:
    raise RuntimeError('Review Cell 2 and set EXECUTE=True before running experiments')
_spec = importlib.util.spec_from_file_location('notebook_workflow', PROJECT / 'scripts/notebook_workflow.py')
_module = importlib.util.module_from_spec(_spec)
_spec.loader.exec_module(_module)
workflow = _module.Workflow(PROJECT, hosted_data, CONFIG, RUN_ID, python=hosted_python,
    device=DEVICE, trust_checkpoint=TRUST_PINNED_DINO, recorded=RUN_RECORDED,
    download_recorded=DOWNLOAD_RECORDED, h5=H5, episodes=EPISODES,
    install=INSTALL_UPDATED_PACKAGE, execute=EXECUTE)

In [ ]:
# Install updated package and validate the original data roster.
setup_done = workflow.setup()

In [ ]:
# Prepare pinned model cache and run actual device/model/simulator preflight.
preflight_done = workflow.preflight(after=setup_done)

In [ ]:
# Experiment 1: LeWM actual-image physical readout.
lewm_readout_done = workflow.diagnostic('lewm', 'readout', after=preflight_done)

In [ ]:
# Experiment 2: LeWM paired prediction vs persistence and zero-future-action controls.
lewm_prediction_done = workflow.diagnostic('lewm', 'prediction-paired', after=lewm_readout_done)

In [ ]:
# Experiment 3: DINO actual-image physical readout.
dino_readout_done = workflow.diagnostic('dinowm', 'readout', after=lewm_prediction_done)

In [ ]:
# Experiment 4: DINO paired prediction vs the same controls.
dino_prediction_done = workflow.diagnostic('dinowm', 'prediction-paired', after=dino_readout_done)

In [ ]:
# Optional recorded data. Explicitly reports SKIPPED when disabled.
recorded_data_done = workflow.prepare_recorded(after=dino_prediction_done)

In [ ]:
# Optional experiment 5: LeWM original-recorded prediction.
lewm_recorded_done = workflow.diagnostic('lewm', 'prediction-recorded', after=recorded_data_done)

In [ ]:
# Optional experiment 6: DINO original-recorded prediction.
dino_recorded_done = workflow.diagnostic('dinowm', 'prediction-recorded', after=lewm_recorded_done)

In [ ]:
# Verify every expected report and artifact checksum. Completion is not a scientific pass.
verified = workflow.finish(after=dino_recorded_done)

In [ ]:
# Inline plots work in Jupyter; Marimo users get the native gallery in the .py notebook.
from IPython.display import Image, display
if verified == 'verified':
    for _directory in workflow.reports:
        print(f'{_directory.parent.name} / {_directory.name}')
        for _image in sorted(_directory.glob('*.png')):
            display(Image(filename=str(_image)))

In [ ]:
# Download this archive through your hosted notebook's file browser.
results_zip = workflow.archive(after=verified)
print(f'Download: {results_zip}')
print('This is a results-only archive. Retain the original input-data and source backup separately.')

## Interpretation and next steps
Inspect readout physical errors and prediction-versus-control confidence intervals before interpreting physical latent edits. All runs on the already-inspected pilot data remain exploratory. Readout validity on actual images is not automatically validity on edited latents. Recorded H5 checks test compatibility with a pretraining distribution, not held-out generalization.

Only after these prerequisites qualify should clean editor competence and a fresh, locked appearance-transfer confirmation be considered. This notebook does not automatically approve those stages. See DIAGNOSTICS.md for protocol details and backup limits.